In [6]:
import numpy as np
import torch
from torch import nn
import pyspiel

In [1]:
GAME_DEF = """universal_poker(
    betting=nolimit,
    bettingAbstraction=fullgame,
    numPlayers=6,
    blind=2 1 0 0 0 0,
    numRounds=4,
    firstPlayer=2 1 1 1,
    numSuits=4,
    numRanks=13,
    numHoleCards=2,
    numBoardCards=0 3 1 1,
    stack=200 200 200 200 200 200
)
"""
GAME_DEF = GAME_DEF.replace("    ", "").replace("\n", "")
GAME_DEF

'universal_poker(betting=nolimit,bettingAbstraction=fullgame,numPlayers=6,blind=2 1 0 0 0 0,numRounds=4,firstPlayer=2 1 1 1,numSuits=4,numRanks=13,numHoleCards=2,numBoardCards=0 3 1 1,stack=200 200 200 200 200 200)'

In [21]:
class ResNet(nn.Module):
    def __init__(
            self, embedding_dim, dropout=0.0, prenorm=True, activation=nn.ReLU()):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            activation,
            nn.Dropout(dropout),
        )
        self.layernorm = nn.LayerNorm(embedding_dim)
        self.prenorm = prenorm
        nn.init.trunc_normal_(self.layer[0].weight, std=0.02, a=-0.04, b=0.04)

    def forward(self, x):
        if self.prenorm:
            return x + self.layer(self.layernorm(x))

        return self.layernorm(x + self.layer(x))


class RNadModel(nn.Module):
    def __init__(self, infostate_tensor_shape, num_actions, hidden_dim, dropout):
        super().__init__()

        self.tower = nn.Sequential(
            nn.Linear(infostate_tensor_shape, hidden_dim),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
        )
        self.policy_tower = nn.Linear(hidden_dim, num_actions)

        self.value_head = nn.Linear(hidden_dim, 1)
        self.log_policy_head = nn.Sequential(
            self.policy_tower,
            nn.LogSoftmax(dim=-1)
        )
        self.policy_head = nn.Sequential(
            self.policy_tower,
            nn.Softmax(dim=-1)
        )


    def forward(self, x):
        embedding = self.tower(x)
        return (
            self.value_head(embedding),
            self.log_policy_head(embedding),
            self.policy_head(embedding)
        )


game = pyspiel.load_game(GAME_DEF)
infostate_tensor_shape = game.information_state_tensor_shape()[0]
num_actions = game.num_distinct_actions()
model = RNadModel(
    infostate_tensor_shape=infostate_tensor_shape,
    num_actions=num_actions,
    hidden_dim=256,
    dropout=0.1
)

state = game.new_initial_state()
while state.is_chance_node():
    actions, probs = zip(*state.chance_outcomes())
    sampled_action = np.random.choice(actions, p=probs)
    state.apply_action(sampled_action)

infromation_state_tensor = torch.tensor(state.information_state_tensor())
model(infromation_state_tensor)

(tensor([0.4980], grad_fn=<ViewBackward0>),
 tensor([-5.3193, -5.2005, -5.1474, -5.6504, -5.7440, -5.4319, -4.7463, -4.9623,
         -5.0291, -5.3513, -5.3099, -5.2210, -4.9611, -5.9141, -5.4449, -5.4846,
         -5.5042, -5.4112, -4.8699, -5.5822, -5.2382, -5.8633, -5.1676, -5.5574,
         -5.4190, -5.6160, -5.0173, -5.3317, -5.1614, -5.2980, -4.5952, -5.2059,
         -5.3559, -5.4727, -5.5577, -5.7115, -5.2822, -5.0765, -5.3783, -5.7715,
         -5.2676, -4.9115, -5.1345, -5.0298, -5.1894, -5.4615, -4.7689, -5.0004,
         -5.0444, -5.4491, -5.8392, -5.6968, -5.2247, -5.3769, -5.4626, -4.7805,
         -5.6721, -5.4164, -5.3161, -5.4600, -5.4959, -5.2734, -5.2957, -5.5187,
         -5.3547, -5.0919, -5.4584, -5.6854, -5.3860, -5.0832, -5.8403, -5.4905,
         -4.8301, -5.8663, -5.4412, -5.7965, -5.6153, -5.3522, -5.5520, -5.3390,
         -4.6060, -4.8912, -5.2230, -5.8434, -5.5150, -5.0868, -5.5603, -5.4419,
         -5.6615, -5.0253, -5.4443, -5.0181, -5.3852, -5.3715, -5

In [20]:
torch.jit.script(model).save("model.pt")